# Notebook Task — Curate Customer Installments

**Objective:** Standardize customer details, convert installment fields to useful data types, and add due-date and payment indicators.

## Why?

**For our project:** Produce typed installment details for queries and reporting.

**For Data Engineering:** Apply consistent data types and run a shared transformation after the selected conditional path succeeds.

#### 01 — Configure Notebook Parameters

Read the catalog and schema parameters to locate the source and output tables.

In [ ]:
# Define defaults for manual execution.
dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.text("schema", "fintech_jobs")

# Read the current parameter values.
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
namespace = f"{catalog}.{schema}"

#### 02 — Standardize and Enrich Installment Details

Convert identifiers, dates, and amounts to appropriate types and standardize text.

Add `due_year`, `due_month`, and `is_paid`. Validate required converted values before writing the curated Delta table.

In [ ]:
from pyspark.sql import functions as F

# Read the joined installment records.
source_df = spark.table(
    f"{namespace}.customers_installments_silver"
)

# Convert data types and standardize text.
typed_df = source_df.select(
    F.expr("try_cast(installment_id AS BIGINT)").alias("installment_id"),
    F.expr("try_cast(application_id AS BIGINT)").alias("application_id"),
    F.expr("try_cast(customer_id AS BIGINT)").alias("customer_id"),
    F.initcap(F.trim(F.col("customer_name"))).alias("customer_name"),
    F.upper(F.trim(F.col("state"))).alias("state"),
    F.expr("try_cast(due_date AS DATE)").alias("due_date"),
    F.expr("try_cast(amount_due AS DECIMAL(12,2))").alias("amount_due"),
    F.upper(F.trim(F.col("payment_status"))).alias("payment_status"),
)

# Stop processing if required converted values are missing.
invalid_rows = typed_df.filter(
    F.col("installment_id").isNull()
    | F.col("application_id").isNull()
    | F.col("customer_id").isNull()
    | F.col("due_date").isNull()
    | F.col("amount_due").isNull()
).count()

print(f"Invalid required values: {invalid_rows}")
assert invalid_rows == 0, "Inspect required values before continuing."

# Enforce one row per installment before writing the curated table.
repeated_ids = (
    typed_df.groupBy("installment_id")
    .count()
    .filter(F.col("count") > 1)
    .take(5)
)

if repeated_ids:
    ids = [row.installment_id for row in repeated_ids]
    raise ValueError(f"Multiple records found for installment IDs {ids}.")

# Add calendar fields and a payment indicator.
curated_df = (
    typed_df
    .withColumn("due_year", F.year("due_date"))
    .withColumn("due_month", F.date_format("due_date", "MM"))
    .withColumn("is_paid", F.col("payment_status") == F.lit("PAID"))
)

# Persist the curated records.
target_table = f"{namespace}.customers_installments_curated"

curated_df.write.format("delta").mode("overwrite").saveAsTable(
    target_table
)

# Inspect the persisted result.
result_df = spark.table(target_table)
result_df.printSchema()

print(f"Curated records: {result_df.count()}")
display(result_df.orderBy("installment_id"))

Invalid required values: 0
root
 |-- installment_id: long (nullable = true)
 |-- application_id: long (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- state: string (nullable = true)
 |-- due_date: date (nullable = true)
 |-- amount_due: decimal(12,2) (nullable = true)
 |-- payment_status: string (nullable = true)
 |-- due_year: integer (nullable = true)
 |-- due_month: string (nullable = true)
 |-- is_paid: boolean (nullable = true)

Curated records: 6


installment_id,application_id,customer_id,customer_name,state,due_date,amount_due,payment_status,due_year,due_month,is_paid
3001,2001,101,Ana Ribeiro,PE,2026-10-20,1750.00,PAID,2026,10,true
3002,2001,101,Ana Ribeiro,PE,2026-11-20,1750.00,PENDING,2026,11,false
3003,2004,104,Pedro Alves,MG,2026-10-25,4000.00,PAID,2026,10,true
3004,2004,104,Pedro Alves,MG,2026-11-25,4000.00,PENDING,2026,11,false
3005,2004,104,Pedro Alves,MG,2026-12-25,4000.00,PENDING,2026,12,false
3006,2005,105,Julia Santos,BA,2026-11-10,2600.00,PENDING,2026,11,false
